# Adaptive Corrective Self-RAG (ACSRAG) Complete Interactive Demo

This notebook demonstrates the enterprise **Adaptive Corrective Self-RAG (ACSRAG)** pipeline combining:
1. **Adaptive Intent Routing**: Automatically classifies queries into `FACTUAL`, `ANALYTICAL`, or `OPINION` modes.
2. **Hybrid Semantic & BM25 Retrieval**: Combines dense vector similarity with Reciprocal Rank Fusion (RRF) and Root Header Boost for candidate/author identity extraction.
3. **Corrective RAG (CRAG) Document Grader**: Strictly evaluates document relevance first (`CORRECT`), routing to Web Search (`INCORRECT (Web Fallback)`) only when the document lacks the concept and Web Search is active.
4. **Self-RAG Reflection & Claim Auditing**: Sentence-level claim extraction, support verification, and calibrated confidence scoring.

In [ ]:
import os

# Set your API keys (or load from .env.local)
os.environ['GOOGLE_API_KEY'] = os.environ.get('GOOGLE_API_KEY', '<YOUR_GOOGLE_API_KEY_HERE>')
os.environ['TAVILY_API_KEY'] = os.environ.get('TAVILY_API_KEY', '<YOUR_TAVILY_API_KEY_HERE>')
print('✅ Environment configured.')

In [ ]:
from pathlib import Path
from acsrag.graphs.phase8_iterative import build_phase8_graph

# Load all documents from the documents directory
pdf_paths = list(Path('documents').glob('*.pdf'))
if not pdf_paths:
    pdf_paths = list(Path('acsrag/documents').glob('*.pdf'))

print(f'Loading {len(pdf_paths)} documents:', [p.name for p in pdf_paths])
graph = build_phase8_graph(pdf_paths)
print('✅ ACSRAG Phase 8 Graph initialized successfully.')

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print('Mermaid graph visualization available in visual environment:', e)

## 📊 ACSRAG Process Trace Visualizer
The helper function below extracts and visualizes the complete step-by-step lifecycle of every query through the ACSRAG pipeline.

In [ ]:
def print_trace(state):
    print("\n" + "="*50)
    print("           ACSRAG PROCESS TRACE")
    print("="*50)
    print(f"USER QUERY: {state.get('question')}\n    ↓")
    if state.get("intent"):
        print(f"Intent Classified: {state['intent']}\n    ↓")
    if state.get("retrieval_query"):
        print(f"Effective Query: {state['retrieval_query']}\n    ↓")
        
    print(f"Vector matches: {len(state.get('docs', []))}")
    print(f"BM25 matches: {len(state.get('bm25_docs', []))}")
    print(f"RRF fused: {len(state.get('fused_docs', []))}\n    ↓")
    
    print(f"CRAG Verdict: {state.get('verdict', 'CORRECT')}")
    print(f"Relevant documents: {len(state.get('good_docs', []))}/{len(state.get('fused_docs', []))}\n    ↓")
    print(f"Context passages: {len(state.get('refined_context', '').split(chr(10)+chr(10))) if state.get('refined_context') else 0}\n    ↓")
    
    claims = state.get("claims", [])
    claim_verdicts = state.get("claim_verdicts", [])
    supported = sum(1 for v in claim_verdicts if v.get("status", "").upper() == "SUPPORTED")
    
    print(f"Supported Claims: {supported} / Unsupported: {len(claims) - supported}\n    ↓")
    print(f"RETRIEVAL REQUIRED: {'YES' if state.get('need_retrieval', True) else 'NO'}\n    ↓")
    
    scores = state.get('confidence_scores', {})
    overall = scores.get('overall_confidence', 'N/A')
    print(f"Overall Confidence: {overall}")
    print("="*50 + "\n")

## 🧪 Multi-Scenario Benchmark Suite
We test 4 key real-world query categories:
1. **Candidate Identity & Contact**: Verified extraction from document header.
2. **Conceptual Domain Comparison (Web Fallback)**: Detecting out-of-document concepts (`LLM vs RAG`) and pivoting to Web Search.
3. **Hybrid Cross-Domain Trend**: Comparing internal skills against live 2026 industry demand.
4. **Multi-Document / Internal Policy Question**: Precise factual retrieval with 0 hallucinations.

In [ ]:
demo_queries = [
    # 1. Candidate Identity from Header (Document-first)
    "What is the name and email address of the candidate in the resume?",
    
    # 2. General Concept (Triggers Web Search fallback when concept is not in resume)
    "LLM vs RAG",
    
    # 3. Hybrid Query (Combines Resume skills + Live Web Search)
    "Compare my AI/ML skills from the resume with current 2026 data science industry demand",
    
    # 4. Academic / Course Slide Comprehension
    "What are the main issues in code generator design according to the lecture slides?"
]

for i, query in enumerate(demo_queries, 1):
    print(f"\n\n{'#'*60}")
    print(f"DEMO QUERY {i}/{len(demo_queries)}: \"{query}\"")
    print(f"{'#'*60}")
    
    final_state = graph.invoke({'question': query, 'iterations': 0})
    print_trace(final_state)
    
    print("FINAL GENERATED ANSWER:")
    print(final_state.get('answer', 'No answer generated.'))